In [0]:
# Silver Layer Transformation

# Objective:
# Clean Bronze data using Data Quality rules and create Silver tables.

# Checks:
# 1. Null Validation
# 2. Duplicate Validation
# 3. Business Rule Validation
# 4. Rejected Records Tracking

In [0]:
customers_bronze = spark.table(
    "banking_catalog.bronze.customers"
)

display(customers_bronze)

customer_id,customer_name,age,city
C101,Rahul Sharma,29,Delhi
C102,Amit Verma,35,Mumbai
C103,Neha Gupta,27,Pune
C104,Priya Singh,31,Bangalore
C105,Arjun Kumar,40,Chennai


In [0]:
# customer_name cannot be null
rejected_customers = customers_bronze.filter(
    customers_bronze.customer_name.isNull()
)

In [0]:
valid_customers = customers_bronze.filter(
    customers_bronze.customer_name.isNotNull()
)

In [0]:
# drop duplicates
valid_customers = valid_customers.dropDuplicates(
    ["customer_id"]
)

In [0]:
#save rejected records 
rejected_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.rejected_customers"
    )

In [0]:
#we are saving silver records
valid_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.customers"
    )

In [0]:
total_records = customers_bronze.count()

valid_records = valid_customers.count()

rejected_records = rejected_customers.count()

print(f"Total Records: {total_records}")
print(f"Valid Records: {valid_records}")
print(f"Rejected Records: {rejected_records}")

Total Records: 5
Valid Records: 5
Rejected Records: 0


In [0]:
#Before we moving to Accounts, I want you to intentionally insert bad data into Bronze.
from pyspark.sql import Row

bad_customer = spark.createDataFrame([
    Row(customer_id="C106",
        customer_name=None,
        age=25,
        city="Delhi")
], customers_bronze.schema)  # Ensure schema matches

#customers_bronze_with_error = customers_bronze.union(bad_customer)
customers_bronze_with_error = customers_bronze.unionByName(bad_customer)


display(customers_bronze_with_error)

customer_id,customer_name,age,city
C101,Rahul Sharma,29,Delhi
C102,Amit Verma,35,Mumbai
C103,Neha Gupta,27,Pune
C104,Priya Singh,31,Bangalore
C105,Arjun Kumar,40,Chennai
C106,null,25,Delhi


In [0]:
customers_bronze_with_error

DataFrame[customer_id: string, customer_name: string, age: int, city: string]

In [0]:
#after injecting bad records

valid_records = valid_customers.count()

rejected_records = rejected_customers.count()

print(f"Total Records: {total_records}")
print(f"Valid Records: {valid_records}")
print(f"Rejected Records: {rejected_records}")

Total Records: 5
Valid Records: 5
Rejected Records: 0


(Accounts Validation)

In [0]:
#Read Bronze Tables
accounts_bronze = spark.table(
    "banking_catalog.bronze.accounts"
)

customers_silver = spark.table(
    "banking_catalog.silver.customers"
)

In [0]:
from pyspark.sql.functions import col

rejected_accounts = accounts_bronze.join(
    customers_silver,
    accounts_bronze.customer_id == customers_silver.customer_id,
    "left_anti"
)

In [0]:
#valid account
valid_accounts = accounts_bronze.join(
    customers_silver,
    accounts_bronze.customer_id == customers_silver.customer_id,
    "inner"
).select(accounts_bronze["*"])

In [0]:
#now we are saving our silver accounts

valid_accounts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.accounts"
    )

In [0]:
#save rejected accounts so that we can troubleshoot in future

rejected_accounts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.rejected_accounts"
    )

In [0]:
print("Total Accounts:", accounts_bronze.count())
print("Valid Accounts:", valid_accounts.count())
print("Rejected Accounts:", rejected_accounts.count())

Total Accounts: 5
Valid Accounts: 5
Rejected Accounts: 0


In [0]:
#for validation we are making a bad account just to insert innormalities 

from pyspark.sql import Row

bad_account = spark.createDataFrame([
    Row(
        account_id="A1006",
        customer_id="C999",
        branch="Delhi",
        balance=50000
    )
])

accounts_test = accounts_bronze.union(bad_account)

In [0]:
valid_accounts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.accounts"
    )

rejected_accounts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.rejected_accounts"
    )

print("Total Accounts:", accounts_test.count())
print("Valid Accounts:", valid_accounts.count())
print("Rejected Accounts:", rejected_accounts.count())


Total Accounts: 6
Valid Accounts: 5
Rejected Accounts: 0


Part 3 - Loans Validation

In [0]:
#lets start with reading of Bronze kayer
loans_bronze = spark.table(
    "banking_catalog.bronze.loans"
)

display(loans_bronze)

#drop duplicates
loans_no_duplicates = loans_bronze.dropDuplicates(
    ["loan_id"]
)

# Why?

# Suppose:

# L003
# L003

# exists twice.
# We keep only one.

loan_id,customer_id,loan_amount,loan_type
L001,C101,500000,Personal
L002,C103,300000,Car
L003,C105,800000,Home


In [0]:
#validate the loan 
#rejected
invalid_loan_amount = loans_no_duplicates.filter(
    loans_no_duplicates.loan_amount <= 0
)

#valid
valid_loan_amount = loans_no_duplicates.filter(
    loans_no_duplicates.loan_amount > 0
)

In [0]:
#every loan must be for valid customers so we need to category them into rejected and accepted 

rejected_loans = valid_loan_amount.join(
    customers_silver,
    valid_loan_amount.customer_id ==
    customers_silver.customer_id,
    "left_anti"
)

#accepted

valid_loans = valid_loan_amount.join(
    customers_silver,
    valid_loan_amount.customer_id ==
    customers_silver.customer_id,
    "inner"
).select(valid_loan_amount["*"])

In [0]:
#Now save the silver loans and rjected loans dseparately
valid_loans.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.loans"
    )

rejected_loans.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.rejected_loans"
    )

In [0]:
print("Total Loans:", loans_bronze.count())
print("Valid Loans:", valid_loans.count())
print("Rejected Loans:", rejected_loans.count())

Total Loans: 3
Valid Loans: 3
Rejected Loans: 0


Part 4 - Transactions Validation

This is the most important banking table.

Business Rules:
Rule 1 : Transaction amount must be positive.

Valid: 5000

Invalid: -5000

Rule 2 : Account must exist.

Rule 3 : Transaction ID should be unique.

In [0]:
#reading the bronze transaction and removing the duplicates one

transactions_bronze = spark.table(
    "banking_catalog.bronze.transactions"
)

transactions_no_duplicates = (
    transactions_bronze
    .dropDuplicates(["transaction_id"])
)

In [0]:
#validate amount by rejecting and accepting

invalid_amount_transactions = (
    transactions_no_duplicates
    .filter(
        transactions_no_duplicates.amount <= 0
    )
)

valid_amount_transactions = (
    transactions_no_duplicates
    .filter(
        transactions_no_duplicates.amount > 0
    )
)

In [0]:
accounts_silver = spark.table(
    "banking_catalog.silver.accounts"
)

In [0]:
# #account validation

# invalid_account_transactions = (
#     valid_amount_transactions.join(
#         spark.table(
#             "banking_catalog.silver.accounts"
#         ),
#         valid_amount_transactions.account_id ==
#         spark.table(
#             "banking_catalog.silver.accounts"
#         ).account_id,
#         "left_anti"
#     )
# )

# valid_transactions = (
#     valid_amount_transactions.join(
#         spark.table(
#             "banking_catalog.silver.accounts"
#         ),
#         valid_amount_transactions.account_id ==
#         spark.table(
#             "banking_catalog.silver.accounts"
#         ).account_id,
#         "inner"
#     ).select(valid_amount_transactions["*"])
# )

In [0]:
invalid_account_transactions = (
    valid_amount_transactions.join(
        accounts_silver,
        valid_amount_transactions.account_id ==
        accounts_silver.account_id,
        "left_anti"
    )
)

valid_transactions = (
    valid_amount_transactions.join(
        accounts_silver,
        valid_amount_transactions.account_id ==
        accounts_silver.account_id,
        "inner"
    )
    .select(valid_amount_transactions["*"])
)

In [0]:
valid_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.transactions"
    )

invalid_amount_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.rejected_transactions"
    )

In [0]:
accounts_silver = spark.table(
    "banking_catalog.silver.accounts"
)

accounts_silver.printSchema()
valid_amount_transactions.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- balance: integer (nullable = true)

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant: string (nullable = true)



In [0]:
print("Total Transactions:", transactions_bronze.count())
print("Valid Transactions:", valid_transactions.count())
print("Rejected Transactions:", invalid_amount_transactions.count())

Total Transactions: 7
Valid Transactions: 7
Rejected Transactions: 0


In [0]:
print(
    "Total Transactions:",
    transactions_bronze.count()
)

print(
    "Valid Transactions:",
    valid_transactions.count()
)

print(
    "Rejected Transactions:",
    invalid_amount_transactions.count()
)

Total Transactions: 7
Valid Transactions: 7
Rejected Transactions: 0


Create Audit Data

In [0]:
from pyspark.sql import Row

audit_data = [

    Row(
        table_name="customers",
        total_records=customers_bronze.count(),
        valid_records=valid_customers.count(),
        rejected_records=rejected_customers.count()
    ),

    Row(
        table_name="accounts",
        total_records=accounts_bronze.count(),
        valid_records=valid_accounts.count(),
        rejected_records=rejected_accounts.count()
    ),

    Row(
        table_name="loans",
        total_records=loans_bronze.count(),
        valid_records=valid_loans.count(),
        rejected_records=rejected_loans.count()
    ),

    Row(
        table_name="transactions",
        total_records=transactions_bronze.count(),
        valid_records=valid_transactions.count(),
        rejected_records=invalid_amount_transactions.count()
    )

]

audit_df = spark.createDataFrame(audit_data)

display(audit_df)

table_name,total_records,valid_records,rejected_records
customers,5,5,0
accounts,5,5,0
loans,3,3,0
transactions,7,7,0


In [0]:
#now we can save the audit table for cross verification
audit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.silver.audit_table"
    )

In [0]:
display(
    spark.table(
        "banking_catalog.silver.audit_table"
    )
)

table_name,total_records,valid_records,rejected_records
customers,5,5,0
accounts,5,5,0
loans,3,3,0
transactions,7,7,0


NOTE : How did we monitor data quality?

You can answer: I created an audit table that captured total, valid, and rejected records for each dataset after Silver layer validation.